# TC3-47 — Analisar diferenças por município, UF, rede e série

**Objetivo:** avaliar como a base e os resultados se comportam nos recortes geográficos e educacionais disponíveis, identificando diferenças entre municípios, unidades federativas (UF), redes de ensino e séries.

## Relação com os notebooks anteriores

- **TC3-43:** fornece o contexto da distribuição do target `alfabetizado` (aproximadamente 51,31% na classe 1 e 48,69% na classe 0, conforme análise anterior).
- **TC3-44 e TC3-45:** reforçam a necessidade de considerar distribuição, valores ausentes, duplicidades e possíveis valores extremos.
- **TC3-46:** identificou associação elevada entre `proficiencia` e `alfabetizado` (correlação aproximada de 0,796), além de associações moderadas com `presenca` e `preenchimento_caderno` (aproximadamente 0,402 cada). Este notebook aprofunda a exploração por grupos, sem tratar associação como causalidade.

## Escopo

1. Ler a base consolidada particionada por UF.
2. Identificar as colunas geográficas e educacionais realmente disponíveis.
3. Comparar quantidade de registros e proporção de `alfabetizado` por UF, rede e série.
4. Investigar diferenças entre municípios, incluindo tamanho da amostra e indicadores médios.
5. Explorar recortes combinados quando as colunas existirem.
6. Registrar achados, limitações e pontos que precisam de validação.

## Cuidados metodológicos

- A base contém milhões de linhas; as comparações devem ser agregadas antes de exibir ou plotar.
- Médias de municípios com poucos registros são instáveis. A tabela de municípios inclui um limite mínimo de observações ajustável.
- Diferenças descritivas não demonstram causalidade e não devem ser interpretadas automaticamente como desempenho atribuível à rede, UF ou município.
- `alfabetizado` é o target recebido pronto na base; este notebook não recalcula sua classificação.
- A associação entre `proficiencia` e `alfabetizado` observada no TC3-46 merece atenção antes da modelagem, mas não comprova, por si só, *data leakage*.

## 1. Preparação do ambiente

O notebook utiliza Polars para consultar os arquivos Parquet de forma eficiente e pandas/matplotlib/seaborn para tabelas e gráficos.

In [ ]:
from pathlib import Path
import warnings

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)
sns.set_theme()

print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)

## 2. Localização e leitura dos arquivos

A estrutura esperada é `dados/consolidado/base_alunos_municipios_por_uf/sigla_uf=SP/00000000.parquet`. O código procura a raiz do projeto a partir do diretório atual e dos diretórios pais.

In [ ]:
def encontrar_raiz_projeto(inicio: Path) -> Path | None:
    inicio = inicio.resolve()
    for candidato in [inicio, *inicio.parents]:
        base = candidato / "dados" / "consolidado" / "base_alunos_municipios_por_uf"
        if base.exists():
            return candidato
    return None

PROJECT_ROOT = encontrar_raiz_projeto(Path.cwd())

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Não encontrei dados/consolidado/base_alunos_municipios_por_uf. "
        "Execute o notebook a partir da raiz do repositório ou ajuste PROJECT_ROOT."
    )

BASE_PATH = PROJECT_ROOT / "dados" / "consolidado" / "base_alunos_municipios_por_uf"
arquivos_parquet = sorted(BASE_PATH.glob("sigla_uf=*/00000000.parquet"))

print("Raiz do projeto:", PROJECT_ROOT)
print("Pasta da base:", BASE_PATH)
print("Quantidade de arquivos Parquet:", len(arquivos_parquet))
print("Exemplos:", [str(p.relative_to(PROJECT_ROOT)) for p in arquivos_parquet[:5]])

if not arquivos_parquet:
    raise FileNotFoundError(f"Nenhum Parquet encontrado em {BASE_PATH}")

lf = pl.scan_parquet(str(BASE_PATH / "sigla_uf=*/00000000.parquet"))
schema = lf.collect_schema()
colunas = schema.names()

print("Quantidade de colunas:", len(colunas))
print("Colunas disponíveis:")
print(colunas)

## 3. Definição das dimensões e indicadores

As colunas são identificadas de forma flexível para evitar assumir nomes que podem variar entre versões da base. A célula mostra quais campos serão utilizados em cada análise.

In [ ]:
def primeira_coluna(candidatas, disponiveis):
    return next((c for c in candidatas if c in disponiveis), None)

COL_UF = primeira_coluna(
    ["sigla_uf", "uf", "nome_uf"], colunas
)
COL_MUNICIPIO = primeira_coluna(
    ["nome_municipio", "nome_do_municipio", "municipio", "id_municipio",
     "codigo_municipio", "cod_municipio", "id_municipio_ibge"], colunas
)
COL_REDE = primeira_coluna(["rede", "tipo_rede", "dependencia_administrativa"], colunas)
COL_SERIE = primeira_coluna(["serie", "ano_escolar", "etapa", "etapa_ensino"], colunas)
COL_ANO = primeira_coluna(["ano_referencia", "ano", "ano_base"], colunas)
COL_TARGET = "alfabetizado" if "alfabetizado" in colunas else None
COL_PROFICIENCIA = "proficiencia" if "proficiencia" in colunas else None
COL_PRESENCA = "presenca" if "presenca" in colunas else None
COL_PREENCHIMENTO = "preenchimento_caderno" if "preenchimento_caderno" in colunas else None

dimensoes = {
    "UF": COL_UF,
    "Município": COL_MUNICIPIO,
    "Rede": COL_REDE,
    "Série": COL_SERIE,
    "Ano de referência": COL_ANO,
}
indicadores = {
    "alfabetizado": COL_TARGET,
    "proficiencia": COL_PROFICIENCIA,
    "presenca": COL_PRESENCA,
    "preenchimento_caderno": COL_PREENCHIMENTO,
}

display(pd.DataFrame([
    {"grupo": grupo, "coluna_detectada": coluna, "disponivel": coluna is not None}
    for grupo, coluna in [*dimensoes.items(), *indicadores.items()]
]))

if COL_TARGET is None:
    print("Atenção: a coluna 'alfabetizado' não foi encontrada; análises de proporção do target serão ignoradas.")

## 4. Visão geral da base

Esta consulta calcula a quantidade de registros e, quando disponível, a distribuição geral do target. O valor é comparado com o contexto observado no TC3-43, mas pode variar conforme a versão e a cobertura da base.

In [ ]:
resumo_geral = lf.select([
    pl.len().alias("quantidade_registros"),
    pl.lit(len(colunas)).alias("quantidade_colunas")
]).collect()
display(resumo_geral.to_pandas())

if COL_TARGET:
    dist_target = (
        lf.group_by(COL_TARGET)
          .agg(pl.len().alias("quantidade"))
          .sort(COL_TARGET)
          .collect()
          .to_pandas()
    )
    dist_target["percentual"] = 100 * dist_target["quantidade"] / dist_target["quantidade"].sum()
    display(dist_target)

## 5. Comparação por UF

Para cada UF, a tabela apresenta o volume de registros e, se as colunas existirem, a proporção média de `alfabetizado`, a proficiência média, a presença e o preenchimento de caderno. A média do target binário é interpretada como proporção da classe 1.

In [ ]:
def construir_agregacao(grupos):
    exprs = [pl.len().alias("quantidade_registros")]
    if COL_TARGET:
        exprs.append(pl.col(COL_TARGET).cast(pl.Float64, strict=False).mean().alias("proporcao_alfabetizado"))
    if COL_PROFICIENCIA:
        exprs.append(pl.col(COL_PROFICIENCIA).cast(pl.Float64, strict=False).mean().alias("proficiencia_media"))
        exprs.append(pl.col(COL_PROFICIENCIA).cast(pl.Float64, strict=False).median().alias("proficiencia_mediana"))
    if COL_PRESENCA:
        exprs.append(pl.col(COL_PRESENCA).cast(pl.Float64, strict=False).mean().alias("presenca_media"))
    if COL_PREENCHIMENTO:
        exprs.append(pl.col(COL_PREENCHIMENTO).cast(pl.Float64, strict=False).mean().alias("preenchimento_caderno_medio"))
    return exprs

def analisar_grupo(coluna, rotulo):
    if coluna is None:
        print(f"Análise por {rotulo} não executada: coluna não encontrada.")
        return None
    resultado = (
        lf.group_by(coluna)
          .agg(construir_agregacao([coluna]))
          .sort("quantidade_registros", descending=True)
          .collect()
          .to_pandas()
    )
    print(f"Quantidade de categorias em {rotulo}: {len(resultado)}")
    display(resultado.head(50))
    return resultado

resumo_uf = analisar_grupo(COL_UF, "UF")

In [ ]:
print(lf.select("sigla_uf").unique().sort("sigla_uf").collect())

## 6. Comparação por rede de ensino

A comparação por rede considera o tamanho de cada grupo. Diferenças na proporção de alfabetização ou na proficiência média são descritivas e podem refletir diferenças de composição, contexto socioeconômico, cobertura ou participação.

In [ ]:
# Agregar os indicadores por código da rede, sem exibir a tabela ainda
resumo_rede = (
    lf.group_by(COL_REDE)
      .agg(construir_agregacao([COL_REDE]))
      .sort("quantidade_registros", descending=True)
      .collect()
      .to_pandas()
)

# Obter a descrição correspondente a cada código
mapa_redes = (
    lf.select("rede", "rede_descricao")
      .unique()
      .collect()
      .to_pandas()
)

# Associar os nomes das redes aos resultados
resumo_rede = resumo_rede.merge(
    mapa_redes,
    on="rede",
    how="left"
)

# Exibir somente a tabela com os nomes das redes
colunas_exibicao = [
    "rede_descricao",
    "quantidade_registros",
    "proporcao_alfabetizado",
    "proficiencia_media",
    "proficiencia_mediana",
    "presenca_media",
    "preenchimento_caderno_medio",
]

display(resumo_rede[colunas_exibicao])

# Gerar o gráfico com os nomes das redes
grafico = resumo_rede.dropna(
    subset=["proporcao_alfabetizado"]
).sort_values(
    "proporcao_alfabetizado",
    ascending=False
)

plt.figure(figsize=(9, 4))
sns.barplot(
    data=grafico,
    x="proporcao_alfabetizado",
    y="rede_descricao"
)
plt.title("Proporção da classe 1 de alfabetizado por rede de ensino")
plt.xlabel("Proporção média do target (0 a 1)")
plt.ylabel("Rede de ensino")
plt.tight_layout()
plt.show()

## 7. Validação do recorte por série

A base consolidada utilizada neste projeto contém registros referentes exclusivamente à série 2, conforme verificado na coluna `serie`.

Como o objetivo do estudo é analisar o índice de alfabetização dos alunos da segunda série, não é necessário realizar uma comparação entre séries. A análise será direcionada às diferenças entre municípios, unidades federativas (UF) e redes de ensino, mantendo a série 2 como recorte fixo.

Essa validação garante que as análises seguintes estejam alinhadas ao escopo do projeto.

In [ ]:
# Validar quais séries estão presentes na base
series_disponiveis = (
    lf.select(COL_SERIE)
      .drop_nulls()
      .unique()
      .sort(COL_SERIE)
      .collect()
)

display(series_disponiveis.to_pandas())

valores_serie = series_disponiveis[COL_SERIE].to_list()

if valores_serie == [2]:
    print("Validação concluída: todos os registros pertencem à série 2.")
else:
    print(
        "Atenção: foram encontrados valores diferentes de 2 "
        "ou mais de uma série na base:",
        valores_serie
    )

## 8. Diferenças entre municípios

A análise municipal usa um limite mínimo de registros para reduzir a instabilidade de grupos muito pequenos. O limite pode ser ajustado conforme o volume observado. A tabela completa de municípios fica disponível para inspeção, e as visualizações mostram apenas os municípios com amostra suficiente.

In [ ]:
MIN_REGISTROS_MUNICIPIO = 30

if COL_MUNICIPIO is None:
    print("Análise municipal não executada: não foi encontrada uma coluna identificadora de município.")
else:
    grupos_municipio = [c for c in [COL_UF, COL_MUNICIPIO] if c is not None]
    resumo_municipio = (
        lf.group_by(grupos_municipio)
          .agg(construir_agregacao(grupos_municipio))
          .sort("quantidade_registros", descending=True)
          .collect()
          .to_pandas()
    )
    print("Municípios/grupos encontrados:", len(resumo_municipio))
    display(resumo_municipio.head(30))

    municipios_validos = resumo_municipio[
        resumo_municipio["quantidade_registros"] >= MIN_REGISTROS_MUNICIPIO
    ].copy()
    print(f"Grupos com pelo menos {MIN_REGISTROS_MUNICIPIO} registros:", len(municipios_validos))

    if COL_TARGET and "proporcao_alfabetizado" in municipios_validos.columns and not municipios_validos.empty:
        colunas_id = [c for c in [COL_UF, COL_MUNICIPIO] if c is not None]
        print("Menores proporções de alfabetizado entre grupos com amostra mínima:")
        display(municipios_validos.sort_values("proporcao_alfabetizado").head(15))
        print("Maiores proporções de alfabetizado entre grupos com amostra mínima:")
        display(municipios_validos.sort_values("proporcao_alfabetizado", ascending=False).head(15))

In [ ]:
display(
    resumo_municipio[
        (resumo_municipio["quantidade_registros"] >= MIN_REGISTROS_MUNICIPIO)
        & (resumo_municipio["proficiencia_media"].notna())
    ]
    .sort_values("proporcao_alfabetizado")
    [
        [
            "sigla_uf",
            "nome_municipio",
            "quantidade_registros",
            "proporcao_alfabetizado",
            "proficiencia_media",
        ]
    ]
    .head(10)
)

In [ ]:
(
    lf.filter(
        (pl.col("sigla_uf") == "GO") &
        (pl.col("nome_municipio") == "Santa Helena de Goiás")
    )
    .group_by("alfabetizado")
    .agg([
        pl.len().alias("quantidade_registros"),
        pl.col("proficiencia").is_null().sum().alias("proficiencia_nula"),
        pl.col("presenca").mean().alias("presenca_media"),
        pl.col("preenchimento_caderno").mean().alias("preenchimento_caderno_medio"),
    ])
    .sort("alfabetizado")
    .collect()
)

In [ ]:
municipios_sem_proficiencia = resumo_municipio[
    resumo_municipio["proficiencia_media"].isna()
]

print(
    "Quantidade de grupos municipais sem proficiência média:",
    len(municipios_sem_proficiencia)
)

display(
    municipios_sem_proficiencia[
        [
            "sigla_uf",
            "nome_municipio",
            "quantidade_registros",
            "proporcao_alfabetizado",
            "proficiencia_media",
            "presenca_media",
            "preenchimento_caderno_medio",
        ]
    ].head(20)
)

In [ ]:
display(
    resumo_municipio[
        (resumo_municipio["proporcao_alfabetizado"] == 0)
        & (resumo_municipio["proficiencia_media"].notna())
    ][
        [
            "sigla_uf",
            "nome_municipio",
            "quantidade_registros",
            "proporcao_alfabetizado",
            "proficiencia_media",
            "presenca_media",
            "preenchimento_caderno_medio",
        ]
    ].head(20)
)

In [ ]:
display(
    resumo_municipio[
        resumo_municipio["quantidade_registros"] >= MIN_REGISTROS_MUNICIPIO
    ]
    .sort_values("proporcao_alfabetizado")
    [
        [
            "sigla_uf",
            "nome_municipio",
            "quantidade_registros",
            "proporcao_alfabetizado",
            "proficiencia_media",
        ]
    ]
    .head(10)
)

In [ ]:
display(
    resumo_municipio[
        (resumo_municipio["quantidade_registros"] >= MIN_REGISTROS_MUNICIPIO)
        & (resumo_municipio["proficiencia_media"].notna())
    ]
    .sort_values("proporcao_alfabetizado", ascending=False)
    [
        [
            "sigla_uf",
            "nome_municipio",
            "quantidade_registros",
            "proporcao_alfabetizado",
            "proficiencia_media",
        ]
    ]
    .head(10)
)

In [ ]:
print(lf.select("alfabetizado").unique().sort("alfabetizado").collect())

In [ ]:
municipios_validos = resumo_municipio[
    (resumo_municipio["quantidade_registros"] >= MIN_REGISTROS_MUNICIPIO)
    & (resumo_municipio["proficiencia_media"].notna())
].copy()

print("Grupos municipais válidos para comparação:", len(municipios_validos))

print(
    "Menor proporção observada:",
    round(municipios_validos["proporcao_alfabetizado"].min() * 100, 2),
    "%"
)

print(
    "Maior proporção observada:",
    round(municipios_validos["proporcao_alfabetizado"].max() * 100, 2),
    "%"
)

print(
    "Mediana das proporções municipais:",
    round(municipios_validos["proporcao_alfabetizado"].median() * 100, 2),
    "%"
)

## 9. Recorte combinado: UF, rede e série

O recorte combinado ajuda a observar se as diferenças encontradas em análises isoladas persistem quando cruzamos as dimensões. Para evitar combinações excessivamente detalhadas, exibimos os grupos com maior volume de registros.

In [ ]:
grupos_combinados = [c for c in [COL_UF, COL_REDE, COL_SERIE] if c is not None]

if len(grupos_combinados) < 2:
    print("Recorte combinado não executado: são necessárias pelo menos duas dimensões disponíveis.")
else:
    resumo_combinado = (
        lf.group_by(grupos_combinados)
          .agg(construir_agregacao(grupos_combinados))
          .sort("quantidade_registros", descending=True)
          .collect()
          .to_pandas()
    )
    print("Quantidade de combinações observadas:", len(resumo_combinado))
    display(resumo_combinado.head(50))

## 10. Recorte por ano de referência, quando disponível

O ano é apresentado como dimensão categórica. Este recorte é uma comparação descritiva por período; a investigação temporal aprofundada será tratada no TC3-48. Mudanças na quantidade de registros entre anos não significam, por si só, erros ou duplicidades: podem refletir coortes, ingresso, saída ou mudanças de cobertura.

In [ ]:
resumo_ano = analisar_grupo(COL_ANO, "ano de referência")

## 11. Síntese automática dos resultados

A síntese abaixo registra os principais grupos observados e as limitações que devem acompanhar a interpretação. Os resultados devem ser revisados após a execução completa do notebook.

In [ ]:
achados = [
    {
        "tema": "Distribuição geral do target",
        "observacao": (
            "A base apresenta aproximadamente 51,31% dos registros "
            "na classe 1 e 48,69% na classe 0."
        ),
        "limitacao": (
            "A variável alfabetizado foi utilizada conforme recebida "
            "na base, sem recalcular a classificação."
        )
    },
    {
        "tema": "Diferenças entre redes de ensino",
        "observacao": (
            "A proporção da classe 1 foi de 51,07% na rede municipal, "
            "53,15% na estadual e 64,00% na privada."
        ),
        "limitacao": (
            "A rede privada possui apenas 25 registros, quantidade "
            "insuficiente para uma comparação robusta."
        )
    },
    {
        "tema": "Diferenças entre municípios",
        "observacao": (
            "Foram identificados 5.444 grupos municipais com pelo menos "
            "30 registros e proficiência média disponível. As proporções "
            "variaram de 7,87% a 100%, com mediana de 56,55%."
        ),
        "limitacao": (
            "As proporções são descritivas e dependem da composição e "
            "da cobertura dos registros de cada município."
        )
    },
    {
        "tema": "Registros sem proficiência",
        "observacao": (
            "Santa Helena de Goiás apresentou 410 registros com "
            "alfabetizado igual a 0, proficiência ausente, presença "
            "igual a 0 e preenchimento do caderno igual a 0."
        ),
        "limitacao": (
            "Não é possível concluir que esses registros representem "
            "alunos não alfabetizados sem investigar a origem dos dados."
        )
    },
    {
        "tema": "Relação com o TC3-46",
        "observacao": (
            "O TC3-46 identificou correlação de aproximadamente 0,796 "
            "entre proficiencia e alfabetizado."
        ),
        "limitacao": (
            "A correlação não comprova causalidade nem data leakage. "
            "A origem e a disponibilidade temporal das variáveis "
            "devem ser verificadas antes da modelagem."
        )
    }
]

display(pd.DataFrame(achados))

## 12. Conclusão

A análise exploratória do TC3-47 permitiu investigar diferenças na proporção de registros classificados como alfabetizados entre unidades federativas, redes de ensino e municípios, considerando a base disponível e o recorte da segunda série.

Na comparação entre redes de ensino, a proporção da classe 1 foi de 51,07% na rede municipal, 53,15% na estadual e 64,00% na privada. Entretanto, a rede privada possui apenas 25 registros, o que limita a confiabilidade dessa comparação.

Na análise municipal, foram identificados 5.444 grupos com pelo menos 30 registros e proficiência média disponível. As proporções observadas variaram de 7,87% a 100%, com mediana de 56,55%. Essa amplitude evidencia diferenças descritivas entre os grupos analisados, mas não permite atribuir suas causas ao município ou a outras características sem investigações adicionais.

Também foi identificado um grupo municipal com 410 registros sem proficiência disponível, presença igual a zero e preenchimento do caderno igual a zero. Esse caso requer investigação da origem dos dados antes de qualquer interpretação sobre alfabetização.

Os resultados devem ser interpretados considerando a cobertura dos dados, a quantidade de registros válidos por grupo e as limitações dos indicadores utilizados. A variável `alfabetizado` foi utilizada conforme recebida na base, sem recalcular sua classificação. Além disso, a relação entre `proficiencia` e `alfabetizado`, observada no TC3-46, deve ser investigada antes da modelagem para avaliar a disponibilidade temporal das informações e o possível risco de *data leakage*.

Conclui-se que o TC3-47 contribui para o entendimento das diferenças observadas na base e fornece subsídios para as próximas etapas do projeto. As comparações realizadas são descritivas e não devem ser interpretadas como comprovação de causalidade nem como reprodução automática do índice oficial de alfabetização.